# 02 — Check old gene IDs against current databases

This notebook checks whether the 369 gene IDs used in 2014 can still be found in today's KEGG, UniProt, and NCBI records. It saves the current database links and flags any gene that cannot be matched clearly, without dropping records or guessing. These results let the final tables connect Liu's original genes to current tools.

Load the database settings and prepare the folders used for downloaded and intermediate data.

In [1]:
import json
import os
import re
import sys
from datetime import date
from pathlib import Path

import pandas as pd
import yaml

sys.path.insert(0, os.path.abspath(".."))
from atlas import crosswalk, fetch
from atlas.schema import RecordOrigin, RecordType

with open("../config/sources.yaml", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)
Path("../data/raw").mkdir(parents=True, exist_ok=True)
Path("../data/interim").mkdir(parents=True, exist_ok=True)
Path("../data/processed").mkdir(parents=True, exist_ok=True)

### Confirm that KEGG has current *A. oryzae* records

Confirm that KEGG's current *A. oryzae* records still use AO090 gene IDs. If KEGG is unavailable, keep the workflow running with UniProt data instead.

In [2]:
kegg_cfg = cfg["sources"]["kegg"]
client = fetch.CachedClient(
    cache_dir="../data/raw",
    rate_limit_seconds=kegg_cfg["rate_limit_seconds"],
)
base = kegg_cfg["base_url"]
org = cfg["organism"]["kegg_org"]
kegg_available = False
kegg_error = None
try:
    kegg_genes = fetch.kegg_gene_list(client, base, org)
    kegg_available = not kegg_genes.empty
    if not kegg_available:
        raise RuntimeError(f"KEGG returned no genes for organism: {org}")
    pattern_rate = kegg_genes["kegg_gene_id"].str.fullmatch(r"aor:AO090\d{9}").mean()
    assert pattern_rate > 0.8, "KEGG IDs no longer support direct AO090 matching"
    print(f"KEGG currently lists {len(kegg_genes):,} A. oryzae genes; {pattern_rate:.1%} use the expected AO090 ID format.")
except Exception as exc:
    kegg_error = f"{type(exc).__name__}: {exc}"
    print("KEGG unavailable; continuing with UniProt-backed identifiers:", kegg_error)
    kegg_genes = pd.DataFrame(columns=["kegg_gene_id", "kegg_description"])

KEGG currently lists 12,102 A. oryzae genes; 99.8% use the expected AO090 ID format.


### Download KEGG links and pathway information

Collect KEGG's links to UniProt, NCBI, functional groups, and pathways so each old ID can be connected to current records. Empty tables are retained when KEGG is unavailable.

In [3]:
if kegg_available:
    kegg_uniprot = fetch.kegg_conv(client, base, org, "uniprot")
    kegg_ncbi = fetch.kegg_conv(client, base, org, "ncbi-geneid")
    kegg_ko_df = fetch.kegg_ko(client, base, org)
    pathways = pd.concat(
        [fetch.kegg_pathway_members(client, base, org, ids, group)
         for group, ids in kegg_cfg["pathways"].items()],
        ignore_index=True,
    )
else:
    kegg_uniprot = pd.DataFrame(columns=["kegg_gene_id", "uniprot_accession"])
    kegg_ncbi = pd.DataFrame(columns=["kegg_gene_id", "ncbi_gene_id"])
    kegg_ko_df = pd.DataFrame(columns=["kegg_gene_id", "kegg_ko"])
    pathways = pd.DataFrame(columns=["kegg_gene_id", "kegg_pathway", "pathway_group"])
kegg_summary = pd.DataFrame({
    "Downloaded KEGG information": ["A. oryzae genes", "Links to UniProt", "Links to NCBI", "Functional-group links", "Relevant pathway memberships"],
    "Rows": [len(kegg_genes), len(kegg_uniprot), len(kegg_ncbi), len(kegg_ko_df), len(pathways)],
})
print("These KEGG tables connect each AO090 gene ID to current database records and pathway information.")
display(kegg_summary)

These KEGG tables connect each AO090 gene ID to current database records and pathway information.


,Downloaded KEGG information,Rows
0,A. oryzae genes,12102
1,Links to UniProt,11624
2,Links to NCBI,12102
3,Functional-group links,4472
4,Relevant pathway memberships,196


### Download current UniProt records

Download the current *A. oryzae* UniProt records and build a table linking AO090 IDs to protein annotations. Multiple candidates are preserved rather than silently reduced to one.

In [4]:
up_cfg = cfg["sources"]["uniprot"]
uniprot_df = fetch.normalise_uniprot(fetch.uniprot_proteome(
    client, up_cfg["base_url"], up_cfg["fields"],
    cfg["organism"]["taxon_id"], cfg["organism"].get("uniprot_proteome"),
))
assert uniprot_df["uniprot_accession"].is_unique
assert len(uniprot_df) > 500, "UniProt response appears truncated to one search page"
print(f"UniProt records downloaded: {len(uniprot_df):,}")
uniprot_df.to_parquet("../data/interim/uniprot_annotations.parquet", index=False)

# A crosswalk is a lookup table connecting the same gene across databases.
# Build it only from explicit AO090 IDs found in each UniProt record.
# Pair NCBI IDs only when their count matches the AO090 IDs; otherwise leave
# the link blank for review instead of guessing which ID belongs to which gene.
locus_rows = []
for row in uniprot_df.itertuples(index=False):
    gene_text = str(getattr(row, "gene_synonyms_raw", "") or "")
    kegg_text = str(getattr(row, "kegg_gene_id_raw", "") or "")
    tags = list(dict.fromkeys(re.findall(r"AO090\d{9}", f"{gene_text} {kegg_text}")))
    ncbi_text = str(getattr(row, "ncbi_gene_id", "") or "")
    ncbi_ids = re.findall(r"\d+", ncbi_text)
    for position, tag in enumerate(tags):
        ncbi_id = ncbi_ids[position] if len(ncbi_ids) == len(tags) else None
        locus_rows.append({
            "ao_locus_tag": tag,
            "kegg_gene_id": f"aor:{tag}" if f"aor:{tag}" in kegg_text else None,
            "uniprot_accession": row.uniprot_accession,
            "ncbi_gene_id": ncbi_id,
            "gene_name": getattr(row, "gene_name", None),
            "function": getattr(row, "function", None),
            "compartment_raw": getattr(row, "compartment_raw", None),
        })
uniprot_locus = pd.DataFrame(locus_rows).drop_duplicates()
print(f"AO090 IDs found in UniProt: {uniprot_locus.ao_locus_tag.nunique():,} "
      f"({len(uniprot_locus):,} gene-to-protein links)")
uniprot_locus.to_parquet("../data/interim/uniprot_locus_crosswalk.parquet", index=False)

No specific UniProt proteome is configured; retrieving all records for the A. oryzae species instead.


UniProt records downloaded: 12,050


AO090 IDs found in UniProt: 12,064 (12,069 gene-to-protein links)


### Match the 2014 gene IDs to current records

Use the current database links to check each Liu AO090 ID while preserving all 369 source rows. Identity is decided from the gene ID itself, not from a similar name or function.

In [5]:
if kegg_available:
    cw = crosswalk.build_crosswalk(kegg_genes, kegg_uniprot, kegg_ncbi, kegg_ko_df)
    identity_lookup = kegg_genes[["kegg_gene_id"]].copy()
    identity_lookup["ao_locus_tag"] = identity_lookup["kegg_gene_id"].str.split(":").str[-1]
    identity_source = "KEGG"
else:
    cw = uniprot_locus.copy()
    identity_lookup = uniprot_locus[["ao_locus_tag", "kegg_gene_id"]].drop_duplicates("ao_locus_tag")
    identity_source = "UniProt AO090 cross-reference"
cw.to_parquet("../data/interim/crosswalk.parquet", index=False)
pathways.to_parquet("../data/interim/kegg_pathways.parquet", index=False)

# Match identity against one row per gene. A gene may have several database
# annotations, but that should not make an exact AO090 ID look ambiguous.
assert identity_lookup["ao_locus_tag"].is_unique

seed = pd.read_csv("../data/interim/liu_components_raw_cleaned.csv")
seed = seed.rename(columns={"ID": "liu_ao_locus_tag"})
assert len(seed) == 369 and seed["liu_ao_locus_tag"].notna().all()
seed = crosswalk.make_record_ids(seed)
resolved = crosswalk.resolve_by_locus_tag(seed, identity_lookup)
crosswalk.assert_no_row_loss(seed, resolved, "record_id")
assert resolved["record_id"].is_unique
resolved["record_origin"] = RecordOrigin.LIU2014.value
resolved["record_type"] = RecordType.MACHINERY.value
mapping_counts = (
    resolved["mapping_status"].value_counts(dropna=False)
    .rename_axis("Result")
    .reset_index(name="Genes")
)
mapping_counts["Meaning"] = mapping_counts["Result"].map({
    "exact": "The 2014 AO090 ID still identifies the same current gene",
    "unresolved": "The ID was not found and remains unassigned",
})
print("ID-matching result for Liu's 369 genes:")
display(mapping_counts)

Some genes have more than one database record: 12102 gene rows became 12103 gene-to-record links. All alternatives are kept for review.


ID-matching result for Liu's 369 genes:


,Result,Genes,Meaning
0,exact,368,The 2014 AO090 ID still identifies the same cu...
1,unresolved,1,The ID was not found and remains unassigned


#### Save the ID-matching results

Write the resolved 369-row table and a summary showing how many IDs matched exactly and how many need review. An exact match means the 2014 AO090 ID still points to the same *A. oryzae* gene in a current database. Unclear rows stay in the table and are marked for manual review rather than matched by a similar name or function.

In [6]:
unresolved = crosswalk.report_unresolved(resolved)
resolved.to_parquet("../data/interim/seed_resolved.parquet", index=False)

report = {
    "retrieved_on": date.today().isoformat(),
    "kegg_available": kegg_available,
    "kegg_error": kegg_error,
    "identity_source": identity_source,
    "kegg_gene_rows": len(kegg_genes),
    "uniprot_rows": len(uniprot_df),
    "crosswalk_rows": len(cw),
    "liu_rows": len(seed),
    "exact_rows": int((resolved.mapping_status == "exact").sum()),
    "flagged_rows": len(unresolved),
    "kegg_ko_links": len(kegg_ko_df),
    "machinery_with_ko": int(seed.liu_ao_locus_tag.isin(kegg_ko_df.kegg_gene_id.str.split(":").str[-1]).sum()),
}
with open("../data/interim/crosswalk_profile.json", "w", encoding="utf-8") as handle:
    json.dump(report, handle, indent=2)
report_summary = pd.DataFrame([
    ("Liu genes checked", report["liu_rows"]),
    ("Exact ID matches", report["exact_rows"]),
    ("Genes needing review", report["flagged_rows"]),
    ("Genes with a KEGG functional group", report["machinery_with_ko"]),
], columns=["Summary", "Count"])
print(f"Current records were retrieved from {report['identity_source']} on {report['retrieved_on']}.")
display(report_summary)

Current records were retrieved from KEGG on 2026-08-07.


,Summary,Count
0,Liu genes checked,369
1,Exact ID matches,368
2,Genes needing review,1
3,Genes with a KEGG functional group,330


### What unresolved IDs mean

If direct matching leaves unresolved rows, inspect them before implementing cross-reference, sequence, or new-orthology resolution. New orthology must remain distinguishable from Liu's assignments.